# PySpark DataFrame Column Operations

## Overview

DataFrame column operations are one of the most important parts of PySpark. They are used to select, reference, rename, create, modify, remove, and transform columns.

Topics covered:

1. `select()` — select columns
2. `col()` — reference columns
3. `alias()` — rename columns in output
4. `withColumn()` — create/modify columns
5. `withColumnRenamed()` — rename existing columns
6. `drop()` — remove columns
7. Column expressions — calculations and transformations
8. Combining multiple column operations

---


# DataFrame DSL

## What is DSL?

**DSL = Domain-Specific Language**

A DSL is a way of writing code that is designed for a particular domain or purpose.

In PySpark, the **DataFrame DSL** refers to using the PySpark DataFrame API to express data transformations using Python code.

For example:

```python
df.filter(
    col("age") > 25
).select(
    "name",
    "age"
).show()
```

This is DataFrame DSL because we are expressing data-processing operations using PySpark's DataFrame API.

The equivalent SQL is:

```sql
SELECT name, age
FROM customers
WHERE age > 25;
```

Both describe a data transformation, but they use different syntax.

---

## Why is it Called a DSL?

The DataFrame API provides operations specifically designed for structured data processing, such as:

```python
select()
filter()
where()
withColumn()
groupBy()
agg()
join()
orderBy()
drop()
```

These operations can be combined to express a data-processing pipeline.

For example:

```python
df \
    .withColumn("annual_salary", col("salary") * 12) \
    .filter(col("age") > 25) \
    .select("name", "annual_salary")
```

Read this as:

```text
Take the DataFrame
        ↓
Create annual_salary
        ↓
Keep rows where age > 25
        ↓
Select name and annual_salary
```

This style of writing transformations is commonly referred to as the **PySpark DataFrame DSL**.

---


```text
DataFrame DSL
      |
      +-- select()
      +-- col()
      +-- alias()
      +-- withColumn()
      +-- withColumnRenamed()
      +-- drop()
      +-- filter()
      +-- where()
      +-- groupBy()
      +-- agg()
      +-- join()
      +-- orderBy()
      +-- ...
```

So the current section is teaching the **DataFrame DSL operation by operation**.

---

## DataFrame DSL vs Spark SQL

Spark provides two common ways to work with structured data.

### 1. DataFrame DSL

```python
df.filter(col("age") > 25) \
  .select("name", "city") \
  .show()
```

### 2. Spark SQL

```python
df.createOrReplaceTempView("customers")

spark.sql("""
    SELECT name, city
    FROM customers
    WHERE age > 25
""").show()
```

Conceptually:

```text
                    Spark
                      |
             +--------+--------+
             |                 |
       DataFrame DSL       Spark SQL
             |                 |
        Python API          SQL syntax
             |                 |
             +--------+--------+
                      |
               Spark Engine
```

Both approaches ultimately describe work that Spark can execute.

---

## DataFrame DSL Methods Covered in This Section

The following methods and concepts are the foundation of the DataFrame DSL:

| Method / Concept | Purpose |
|---|---|
| `select()` | Select columns |
| `col()` | Reference columns |
| `alias()` | Rename an expression/output column |
| `withColumn()` | Create or modify a column |
| `withColumnRenamed()` | Rename an existing column |
| `drop()` | Remove columns |
| Column expressions | Perform calculations and transformations |
| `when()` | Create conditional expressions |
| `otherwise()` | Define the alternative condition |
| `lit()` | Create constant/literal values |
| Combining operations | Build transformation pipelines |

### Mental Model

```text
DataFrame
    |
    ↓
DataFrame DSL
    |
    +-- Select
    +-- Reference
    +-- Rename
    +-- Create
    +-- Modify
    +-- Drop
    +-- Calculate
    +-- Apply conditions
    |
    ↓
Transformation Pipeline
    |
    ↓
Spark Execution Engine
```

---

# 1. `select()` — Select Columns

## Definition

`select()` is used to choose specific columns from a DataFrame.

### Example DataFrame

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ColumnOperations") \
    .getOrCreate()

data = [
    ("C001", "Karthik", "Chennai", 25, 55000.50),
    ("C002", "Arun", "Mumbai", 28, 62000.75),
    ("C003", "Rahul", "Delhi", 26, 48000.25)
]

columns = ["customer_id", "name", "city", "age", "salary"]

df = spark.createDataFrame(data, columns)

df.show()
```

---

## Select One Column

```python
df.select("name").show()
```

Output:

```text
+-------+
|   name|
+-------+
|Karthik|
|   Arun|
|  Rahul|
+-------+
```

---

## Select Multiple Columns

```python
df.select("name", "city", "age").show()
```

Equivalent SQL:

```sql
SELECT name, city, age
FROM customers;
```

---

## Select All Columns

```python
df.select("*").show()
```

This selects every column.

---

## Important Point

`select()` does not modify the original DataFrame.

```python
df2 = df.select("name", "city")
```

Now:

```text
df  → original DataFrame
df2 → DataFrame containing only name and city
```

### Mental Model

```text
DataFrame
    |
    | select()
    ↓
Selected DataFrame
```

---

# 2. `col()` — Reference Columns

## Definition

`col()` is used to reference a DataFrame column inside PySpark expressions.

Import it using:

```python
from pyspark.sql.functions import col
```

Then:

```python
df.select(col("name")).show()
```

This is similar to:

```python
df.select("name").show()
```

Both select the `name` column.

---

## Why Do We Need `col()`?

`col()` becomes especially useful when we want to perform operations on columns.

For example:

```python
df.select(
    col("name"),
    col("salary") + 5000
).show()
```

Here:

```python
col("salary")
```

means:

> Reference the `salary` column.

Then:

```python
col("salary") + 5000
```

means:

> Take the salary value and add 5000.

---

## Column Expressions

```python
df.select(
    col("name"),
    col("age") + 1
).show()
```

The expression:

```python
col("age") + 1
```

creates a new calculated expression.

---

## Mental Model

```text
col("salary")
      ↓
Reference salary column
      ↓
col("salary") + 5000
      ↓
Column expression
```

---

# 3. `alias()` — Rename Columns in Output

## Definition

`alias()` gives a temporary name to a column or column expression in the resulting DataFrame.

Example:

```python
df.select(
    col("name").alias("customer_name")
).show()
```

The output column will be:

```text
customer_name
```

instead of:

```text
name
```

---

## Alias a Calculated Column

```python
df.select(
    col("name"),
    (col("salary") + 5000).alias("new_salary")
).show()
```

Here:

```python
(col("salary") + 5000).alias("new_salary")
```

means:

1. Take salary
2. Add 5000
3. Give the resulting column the name `new_salary`

---

## Multiple Aliases

```python
df.select(
    col("customer_id").alias("id"),
    col("name").alias("customer_name"),
    col("city").alias("customer_city")
).show()
```

---

## `alias()` vs `withColumnRenamed()`

`alias()` is commonly used when selecting or creating an expression:

```python
df.select(
    col("name").alias("customer_name")
)
```

`withColumnRenamed()` is used to rename an existing DataFrame column:

```python
df.withColumnRenamed(
    "name",
    "customer_name"
)
```

---

# 4. `withColumn()` — Create or Modify Columns

## Definition

`withColumn()` is used to:

- Create a new column
- Modify an existing column

Syntax:

```python
df.withColumn("new_column", expression)
```

---

## Create a New Column

Example:

```python
df2 = df.withColumn(
    "age_plus_5",
    col("age") + 5
)
```

Now the DataFrame contains:

```text
customer_id
name
city
age
salary
age_plus_5
```

---

## Another Example

```python
df2 = df.withColumn(
    "annual_salary",
    col("salary") * 12
)
```

This creates a new column called:

```text
annual_salary
```

---

## Modify an Existing Column

`withColumn()` can also replace an existing column.

```python
df2 = df.withColumn(
    "salary",
    col("salary") + 5000
)
```

The existing `salary` column is replaced by the new expression.

---

## Add a Constant Value

```python
from pyspark.sql.functions import lit

df2 = df.withColumn(
    "country",
    lit("India")
)
```

`lit()` creates a literal/constant value.

Output conceptually:

```text
name       country
Karthik    India
Arun       India
Rahul      India
```

---

## Important Point

Like other DataFrame transformations, `withColumn()` does not modify `df` in place.

```python
df2 = df.withColumn("age_plus_5", col("age") + 5)
```

The original:

```python
df
```

remains unchanged.

---

# 5. `withColumnRenamed()` — Rename Existing Columns

## Definition

`withColumnRenamed()` is used to rename an existing DataFrame column.

Syntax:

```python
df.withColumnRenamed("old_name", "new_name")
```

Example:

```python
df2 = df.withColumnRenamed(
    "name",
    "customer_name"
)
```

Now:

```text
name
```

becomes:

```text
customer_name
```

---

## Rename Multiple Columns

You can chain the operation:

```python
df2 = df \
    .withColumnRenamed("name", "customer_name") \
    .withColumnRenamed("city", "customer_city")
```

---

## Difference Between `alias()` and `withColumnRenamed()`

### `alias()`

Used mainly while selecting or creating expressions:

```python
df.select(
    col("name").alias("customer_name")
)
```

### `withColumnRenamed()`

Used to rename an existing DataFrame column:

```python
df.withColumnRenamed(
    "name",
    "customer_name"
)
```

### Mental Model

```text
alias()
    ↓
Rename expression/output

withColumnRenamed()
    ↓
Rename existing DataFrame column
```

---

# 6. `drop()` — Remove Columns

## Definition

`drop()` is used to remove one or more columns from a DataFrame.

---

## Drop One Column

```python
df2 = df.drop("age")
```

The `age` column is removed.

---

## Drop Multiple Columns

```python
df2 = df.drop(
    "age",
    "salary"
)
```

Now both columns are removed.

---

## Example

Original:

```text
customer_id
name
city
age
salary
```

After:

```python
df2 = df.drop("age", "salary")
```

Result:

```text
customer_id
name
city
```

---

## Important Point

`drop()` does not modify the original DataFrame.

```python
df2 = df.drop("age")
```

The original `df` still contains `age`.

---

# 7. Column Expressions — Calculations and Transformations

## Definition

A column expression is an operation performed on one or more DataFrame columns.

Column expressions are commonly created using:

```python
col()
```

Examples:

```python
col("salary") + 5000
```

```python
col("salary") * 12
```

```python
col("age") + 1
```

---

## Arithmetic Expressions

### Addition

```python
df.select(
    col("salary") + 5000
).show()
```

### Subtraction

```python
df.select(
    col("salary") - 5000
).show()
```

### Multiplication

```python
df.select(
    col("salary") * 12
).show()
```

### Division

```python
df.select(
    col("salary") / 12
).show()
```

---

## Give Calculated Columns a Name

Without alias:

```python
df.select(
    col("salary") * 12
).show()
```

With alias:

```python
df.select(
    (col("salary") * 12).alias("annual_salary")
).show()
```

The second version is clearer because the output column has a meaningful name.

---

## Multiple Column Expressions

```python
df.select(
    col("name"),
    col("age") + 1,
    col("salary") * 12
).show()
```

Better:

```python
df.select(
    col("name"),
    (col("age") + 1).alias("next_year_age"),
    (col("salary") * 12).alias("annual_salary")
).show()
```

---

## Conditional Expressions

PySpark also supports conditional column expressions using `when()` and `otherwise()`.

Import:

```python
from pyspark.sql.functions import when
```

Example:

```python
df.select(
    col("name"),
    when(col("age") >= 27, "Senior")
    .otherwise("Junior")
    .alias("age_category")
).show()
```

Conceptually:

```text
age >= 27
    |
    +---- True  → Senior
    |
    +---- False → Junior
```

This is similar to SQL:

```sql
CASE
    WHEN age >= 27 THEN 'Senior'
    ELSE 'Junior'
END
```

---

## Conditional Column with `withColumn()`

```python
df2 = df.withColumn(
    "age_category",
    when(col("age") >= 27, "Senior")
    .otherwise("Junior")
)
```

---

# 8. Combining Multiple Column Operations

In real Data Engineering pipelines, column operations are usually combined.

Consider the original DataFrame:

```text
customer_id | name    | city    | age | salary
------------|---------|---------|-----|--------
C001        | Karthik | Chennai | 25  | 55000.50
C002        | Arun    | Mumbai  | 28  | 62000.75
C003        | Rahul   | Delhi   | 26  | 48000.25
```

---

## Example 1 — Select + Alias

```python
from pyspark.sql.functions import col

df2 = df.select(
    col("customer_id").alias("id"),
    col("name").alias("customer_name"),
    col("city")
)

df2.show()
```

---

## Example 2 — Select + Calculated Column

```python
df2 = df.select(
    col("name"),
    col("city"),
    (col("salary") * 12).alias("annual_salary")
)

df2.show()
```

---

## Example 3 — `withColumn()` + `withColumn()`

```python
df2 = df \
    .withColumn("annual_salary", col("salary") * 12) \
    .withColumn("age_plus_5", col("age") + 5)

df2.show()
```

---

## Example 4 — Rename + Create + Drop

```python
df2 = df \
    .withColumnRenamed("name", "customer_name") \
    .withColumn("annual_salary", col("salary") * 12) \
    .drop("age")

df2.show()
```

Flow:

```text
Original DataFrame
        |
        ↓
Rename name
        |
        ↓
Create annual_salary
        |
        ↓
Drop age
        |
        ↓
Final DataFrame
```

---

## Example 5 — Complete Transformation

```python
from pyspark.sql.functions import col, lit, when

df2 = df \
    .withColumnRenamed("name", "customer_name") \
    .withColumn(
        "annual_salary",
        col("salary") * 12
    ) \
    .withColumn(
        "age_category",
        when(col("age") >= 27, "Senior")
        .otherwise("Junior")
    ) \
    .withColumn(
        "country",
        lit("India")
    ) \
    .drop("age")

df2.show()
```

This performs several transformations:

```text
name
 ↓
customer_name

salary
 ↓
annual_salary

age
 ↓
age_category

constant
 ↓
country

age
 ↓
removed
```

---

# Important Mental Model

Think of DataFrame column operations as building a transformation pipeline.

```text
                 DataFrame
                     |
          +----------+----------+
          |          |          |
       select()    col()     drop()
          |          |
          |       expression
          |          |
          +----+-----+
               |
         withColumn()
               |
          new/modified
            column
               |
      withColumnRenamed()
               |
          renamed column
               |
            alias()
               |
        output name
```

---

# Quick Reference

| Operation | Purpose |
|---|---|
| `select()` | Select columns |
| `col()` | Reference a column |
| `alias()` | Rename an expression/output column |
| `withColumn()` | Create or modify a column |
| `withColumnRenamed()` | Rename an existing column |
| `drop()` | Remove columns |
| `when()` | Create conditional expressions |
| `otherwise()` | Define the alternative condition |
| `lit()` | Create a constant/literal column |

---

# SQL vs PySpark Comparison

### Select

SQL:

```sql
SELECT name, city
FROM customers;
```

PySpark:

```python
df.select("name", "city")
```

### Calculated Column

SQL:

```sql
SELECT salary * 12 AS annual_salary
FROM customers;
```

PySpark:

```python
df.select(
    (col("salary") * 12).alias("annual_salary")
)
```

### Rename

SQL:

```sql
SELECT name AS customer_name
FROM customers;
```

PySpark:

```python
df.select(
    col("name").alias("customer_name")
)
```

### Conditional Column

SQL:

```sql
SELECT
    CASE
        WHEN age >= 27 THEN 'Senior'
        ELSE 'Junior'
    END AS age_category
FROM customers;
```

PySpark:

```python
df.select(
    when(col("age") >= 27, "Senior")
    .otherwise("Junior")
    .alias("age_category")
)
```

---

# Key Takeaways

1. `select()` chooses columns.
2. `col()` references columns and is essential for expressions.
3. `alias()` gives a new name to an expression/output column.
4. `withColumn()` creates or replaces a column.
5. `withColumnRenamed()` renames an existing column.
6. `drop()` removes columns.
7. Column expressions allow calculations and transformations.
8. `when()` and `otherwise()` are useful for conditional transformations.
9. DataFrame transformations return new DataFrames rather than modifying the original DataFrame.
10. Multiple transformations can be chained to build an ETL pipeline.

## Core Pattern to Remember

```python
df2 = df \
    .select(...) \
    .withColumn(...) \
    .withColumnRenamed(...) \
    .drop(...)
```

The exact combination depends on the transformation requirement.
